# 09 - Reshape into player A versus player B
**What it does:** <in your own words, one or two sentences>
**Needs:** notebook 05 has run (tennis_clean.matches_dated exists)
**Produces:** the table tennis_clean.matches_ab, and a Parquet snapshot of it in the bucket under processed/
**Run after:** 05_match_dates (it does not need the weather notebooks)

## Install the pinned libraries

In [ ]:
# The repo is public, so requirements.txt can be read straight from GitHub.
REQUIREMENTS_URL = "https://raw.githubusercontent.com/katiedevane/tennis-project/main/requirements.txt"
!pip install -q -r $REQUIREMENTS_URL

## Sign in and load settings from Colab Secrets
Switch on Notebook access for both secrets in this notebook.

In [ ]:
import os
from google.colab import auth, userdata

auth.authenticate_user()

PROJECT_ID = userdata.get("TENNIS_PROJECT_ID")
BUCKET_NAME = userdata.get("TENNIS_BUCKET_NAME")
REGION = "EU"        # the bucket and the BigQuery datasets must share this location
DATASETS = ["tennis_raw", "tennis_clean", "tennis_features"]
os.environ["CLOUDSDK_CORE_PROJECT"] = PROJECT_ID

assert PROJECT_ID and BUCKET_NAME, "A secret is empty: check its name and Notebook access"
print("Settings loaded from Colab Secrets.")

## Helpers

In [ ]:
import pandas as pd
from google.cloud import bigquery, storage

bq = bigquery.Client(project=PROJECT_ID, location=REGION)
DATED = f"{PROJECT_ID}.tennis_clean.matches_dated"
AB = f"{PROJECT_ID}.tennis_clean.matches_ab"

def run(sql):
    config = bigquery.QueryJobConfig(maximum_bytes_billed=1_000_000_000)   # safety cap: 1 GB per query
    return bq.query(sql, job_config=config).to_dataframe()

pd.set_option("display.max_rows", 200, "display.width", 200)

## Build the table
The raw data has a winner side and a loser side, so building features from it would build the label into the layout. Each match is reshaped into **player A versus player B**, with the target `a_won` (1 if A won, else 0).

**Which player is A** is decided by a hash of the match ID. It is deterministic (a rerun gives the same table) and has nothing to do with who won or who is ranked higher. The salt in the hash can be changed to redraw the assignment.

**Columns known only after the match** (score, minutes, serve statistics, walkover, retirement and default flags) get the prefix `post_`. A column without that prefix is known before the match. `post_` columns can only be used as history for later matches, never as features of the same match.

In [ ]:
SALT = "ab-v1"     # part of the assignment rule; changing it redraws A/B

CONTEXT = ["match_id", "tourney_id", "tourney_name", "name_key", "tourney_date", "season",
           "tourney_level", "surface", "draw_size", "best_of", "round", "event_format",
           "match_date_min", "match_date_max"]
PLAYER_FIELDS = ["id", "name", "seed", "entry", "hand", "ht", "ioc", "age", "rank", "rank_points"]
STAT_FIELDS = ["ace", "df", "svpt", "1stIn", "1stWon", "2ndWon", "SvGms", "bpSaved", "bpFaced"]

parts = []
for f in PLAYER_FIELDS:                                   # known before the match
    parts.append(f"IF(a_is_winner, winner_{f}, loser_{f}) AS a_{f}")
    parts.append(f"IF(a_is_winner, loser_{f}, winner_{f}) AS b_{f}")
for f in STAT_FIELDS:                                     # only known after the match
    parts.append(f"IF(a_is_winner, w_{f}, l_{f}) AS post_a_{f}")
    parts.append(f"IF(a_is_winner, l_{f}, w_{f}) AS post_b_{f}")

sql = f"""
CREATE OR REPLACE TABLE `{AB}`
PARTITION BY DATE_TRUNC(tourney_date, MONTH)
CLUSTER BY tourney_id
OPTIONS (description = 'One row per match: player A vs player B (A chosen by hash of match_id), label a_won. Columns starting post_ are only known after the match.')
AS
WITH base AS (
  SELECT *, MOD(FARM_FINGERPRINT(CONCAT('{SALT}|', match_id)), 2) = 0 AS a_is_winner
  FROM `{DATED}`
)
SELECT {", ".join(CONTEXT)},
       {", ".join(parts)},
       IF(a_is_winner, 1, 0) AS a_won,
       score AS post_score,
       minutes AS post_minutes,
       is_walkover AS post_is_walkover,
       is_retirement AS post_is_retirement,
       is_default AS post_is_default
FROM base
"""
job = bq.query(sql); job.result()
print("Built", AB, "| bytes processed:", f"{job.total_bytes_processed:,}")

## Check: does the new table match its source?
Same number of rows, no winner or loser columns left, and every A/B value traces back to the winner or loser value it came from (both counts at the end must be 0).

In [ ]:
cols = [f.name for f in bq.get_table(AB).schema]
print("columns:", len(cols))
print("winner/loser columns left:", [c for c in cols if "winner" in c or "loser" in c])     # expect []
print("post_ columns:", [c for c in cols if c.startswith("post_")])

print(run(f"""SELECT (SELECT COUNT(*) FROM `{AB}`) AS ab_rows,
                     (SELECT COUNT(*) FROM `{DATED}`) AS dated_rows,
                     (SELECT COUNT(DISTINCT match_id) FROM `{AB}`) AS unique_match_ids"""))

checks = [f"(a.a_{f} IS NOT DISTINCT FROM IF(a.a_won = 1, m.winner_{f}, m.loser_{f}) "
          f"AND a.b_{f} IS NOT DISTINCT FROM IF(a.a_won = 1, m.loser_{f}, m.winner_{f}))" for f in PLAYER_FIELDS]
print(run(f"""
SELECT COUNTIF(NOT ({" AND ".join(checks)})) AS inconsistent_rows,
       COUNTIF(a.a_id = a.b_id) AS same_player_both_sides
FROM `{AB}` a JOIN `{DATED}` m USING (match_id)
"""))

## Check: is the A/B assignment balanced and independent of the result?
- The share of matches won by A should be about 0.5 overall (small seasons can drift a little by chance).
- Player A should be the better-ranked player about as often as player B is.
- The better-ranked player should win clearly more than half the time, and about equally as A and as B. That is real tennis signal appearing equally on both sides, which shows the columns are oriented correctly.

In [ ]:
print(run(f"SELECT COUNT(*) AS n, ROUND(AVG(a_won), 4) AS share_a_won FROM `{AB}`"))
print(run(f"SELECT season, COUNT(*) AS n, ROUND(AVG(a_won), 3) AS share_a_won FROM `{AB}` GROUP BY 1 ORDER BY 1"))
print(run(f"SELECT tourney_level, surface, COUNT(*) AS n, ROUND(AVG(a_won), 3) AS share_a_won FROM `{AB}` GROUP BY 1, 2 ORDER BY 1, 2"))

print(run(f"""
SELECT COUNTIF(a_rank < b_rank) AS a_better_ranked,
       COUNTIF(a_rank > b_rank) AS b_better_ranked,
       ROUND(AVG(IF(a_rank < b_rank, a_won, NULL)), 3) AS better_ranked_wins_when_A,
       ROUND(AVG(IF(a_rank > b_rank, 1 - a_won, NULL)), 3) AS better_ranked_wins_when_B
FROM `{AB}` WHERE a_rank IS NOT NULL AND b_rank IS NOT NULL
"""))

## Save a Parquet snapshot and a manifest
A snapshot of the reshaped table before features exist (Parquet keeps the data types). If a snapshot for this version already exists it is not overwritten; changes go to a new version.

In [ ]:
import json, hashlib
from datetime import datetime, timezone

bucket = storage.Client(project=PROJECT_ID).bucket(BUCKET_NAME)
EXPORT_PREFIX = "processed/matches_ab/v1/"

if any(True for _ in bucket.list_blobs(prefix=EXPORT_PREFIX, max_results=1)):
    print("v1 export already exists; not overwriting. Save any change as v2.")
else:
    bq.extract_table(AB, f"gs://{BUCKET_NAME}/{EXPORT_PREFIX}part-*.parquet",
                     job_config=bigquery.ExtractJobConfig(destination_format="PARQUET")).result()

files = [{"file": b.name, "bytes": b.size, "sha256": hashlib.sha256(b.download_as_bytes()).hexdigest()}
         for b in bucket.list_blobs(prefix=EXPORT_PREFIX)]
rows = run(f"SELECT COUNT(*) AS n FROM `{AB}`")["n"][0]
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
bucket.blob(f"manifests/matches_ab_v1_{stamp}.json").upload_from_string(
    json.dumps({"table": "tennis_clean.matches_ab", "rows": int(rows), "salt": SALT, "files": files}, indent=1),
    content_type="application/json")
print(files)